---
# Chapter 11 — Consequences Nobody Wrote Down

## Orientation

| Field | Value |
|-------|-------|
| Chapter | 11: Consequences Nobody Wrote Down |
| Central question | How can memory detect unfinished consequences nobody wrote down without inventing obligations? |
| Main concepts | Derived loop, Triple gate, Inferred precision |
| Implementation | derived_loops |
| Experiment | ch11-20260920T165907Z-derived-loops |
| Evidence status | Book result: conditional / guarded |
| Depends on | Chapter 9 (open loops), Chapter 7 (lineage) |

---


## What this notebook demonstrates

Beyond stated tasks lie consequences nobody wrote down — and a permissive reader invents them freely. The notebook:

1. **Runs the staged triple gate** (`check_candidate`/`apply_gate`): current, expected and difference legs present, traceable, stated-not-inferred, fresh, uncancelled
2. **Scores one scope task** against the ledger (`ledger_genuine` vs `ledger_harmful`)
3. **Ablates each gate leg** to show what the permissive alternative admits
4. **Loads the frozen contrast**: unconstrained precision 0.458 with a 0.50 harmful-task rate vs staged 0.875 with zero harm

> **Evidence status**: Book result, conditional and guarded. The staged gate matched the controlled oracle on crisp synthetic fixtures; that crispness limits the claim.


## The chapter question

> **What consequences are implied but unwritten?**

A derived obligation is useful only when current state, expected state, difference and cancelling evidence are all treated as separate licence conditions. Remove any leg and the backtest must fail — that is what makes the gate earned rather than cautious.


## Concepts in this chapter


In [ ]:
import sys
from pathlib import Path


def _find_repo_root(start):
    cur = Path(start).resolve()
    while True:
        if ((cur / "content").is_dir() and (cur / "notebooks").is_dir()
                and (cur / "solution").is_dir()):
            return cur
        if cur == cur.parent:
            raise RuntimeError("could not locate repository root")
        cur = cur.parent


REPO_ROOT = _find_repo_root(Path.cwd())
for _p in (str(REPO_ROOT), str(REPO_ROOT / "solution")):
    if _p not in sys.path:
        sys.path.insert(0, _p)

from notebooks.memory._support import load_chapter_metadata, render_table

meta = load_chapter_metadata(11)
concepts = (meta.get("chapter", {}).get("concepts")
            or meta.get("concepts", []))
render_table([
    {"Concept ID": c["id"], "Name": c["name"], "Status": c["status"]}
    for c in concepts
], "Chapter 11 Concepts")

## The running example

Six derived candidates over one scope: genuine completions (`c-docs-flags`, `c-cli-half`, `c-web`) alongside traps — a facade deletion cancelled by contract, a test removal under intentional preservation, a docs change under deferral. The ledger marks which listings would harm if acted on.


## The mechanism: the staged gate

S0 three legs present. S1 each cited leg traceable to a raw artifact. S2 the expected-state leg stated, never inferred. S3 the current-state leg fresh as of the standpoint. S4 no cancelling evidence. Reason codes record the algorithm's path; the `skip_*` flags exist only to run ablations, never in production.


In [ ]:
from derived_loops import fixtures as FX
from derived_loops import gate as G
from derived_loops import metrics as M

tasks = FX.all_tasks()
print(f"Scope tasks: {len(tasks)}")
task = tasks[0]
print(f"Task {task.task_id}: {len(task.candidates)} candidates, "
      f"standpoint {task.standpoint}")

decisions = G.apply_gate(task)
for d in decisions:
    print(f"  {d.candidate_id:16s} admitted={d.admitted!s:5s} "
          f"stage={d.stage_failed or '-':4s} {d.reason}")
print("\nLedger genuine:", sorted(FX.ledger_genuine(task)))
print("Ledger harmful:", sorted(FX.ledger_harmful(task)))
score = M.score_task(task, decisions)
print(f"Score: TP={score['true_positives']} FP={score['false_positives']} "
      f"FN={score['false_negatives']} harmful_listed={score['harmful_listed']}")

In [ ]:
# Ablation: skip each licence leg in turn. The permissive alternative
# produces harmful listings; removing a gate leg fails the backtest.
for label, kw in [("full gate", {}),
                  ("skip current freshness", {"skip_current": True}),
                  ("skip stated-expected", {"skip_expected": True}),
                  ("skip cancelling check", {"skip_cancelling": True})]:
    ds = G.apply_gate(task, **kw)
    bad = [d.candidate_id for d in ds if d.admitted
           and d.candidate_id in FX.ledger_harmful(task)]
    print(f"{label:24s} admitted={[d.candidate_id for d in ds if d.admitted]} "
          f"harmful={bad}")

In [ ]:
from notebooks.memory._support import load_frozen_run

metrics = load_frozen_run("ch11-20260920T165907Z-derived-loops")["metrics"]
render_table(
    [{"Condition": c,
      "Inferred precision": m["mean_inferred_precision"],
      "Inferred recall": m["mean_inferred_recall"],
      "Harmful-task rate": m["harmful_task_rate"],
      "Evidence correctness": m["evidence_correctness"]}
     for c, m in metrics.items() if c != "backtest"],
    "Frozen ch11 conditions (8 tasks each)")

## What happened?

The unconstrained listing admits harmful obligations at a 0.50 task rate with precision 0.458. The staged gate removes every harmful listing while holding recall at 0.875 and reaching evidence correctness 1.0. On this scope task the cancelling leg carries the load (run the ablation cell: only skipping S4 re-admits harm); the frozen backtest across all 8 tasks is what shows every leg biting somewhere — the licence is load-bearing, not decorative.


## Connect this to the experiment

The frozen run adds the oracle ceiling, the abstain condition and the backtest discipline: a candidate gate version replays over every task and promotes only on primary-metric gain with no gate breached. Crisp synthetic fixtures limit how far the claim travels; that limit is part of the result.


## What this establishes

- **Derived consequences need a staged licence**, not a scalar threshold
- **Each removed leg re-admits harm** (backtest-verified)
- **Guarded and conditional**: synthetic crispness bounds the claim


## What this does NOT establish

- Obligation discovery in messy, real-world prose
- Recall of consequences the ledger never listed
- A general consequence engine


In [ ]:
# TRY IT YOURSELF: gate a single candidate with one leg removed and
# read the exact stage that refuses it.
cand = task.candidates[1]
for kw in ({}, {"skip_cancelling": True}):
    d = G.check_candidate(cand, task.standpoint, **kw)
    print(f"{str(sorted(kw)) :28s} -> admitted={d.admitted} "
          f"stage={d.stage_failed} ({d.reason})")

## Where this leads next

Chapter 12 puts the whole pipeline to the behavioural test: same reader, same task, memory varied, behaviour compared.

> **See this chapter in code:** [Open the companion Jupyter notebook](memory\11-chapter.ipynb)
